# Illogical transitions
- This is a comparison within several vector files between years


In [1]:
import os
import geopandas as gpd
import rasterio
from shapely.geometry import LineString, Polygon
from difflib import SequenceMatcher
import numpy as np
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import from_origin
from rasterio.crs import CRS

In [2]:
path = r"Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat"
illogical_transitions_csv = r""

In [14]:
def get_vector_file_list(path):
    """
    Get a list of the vector files inside the folder
    """
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        # "anat" is just to get here necessary ones
        if file.endswith("anat.shp"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def update_names_based_on_similarity(unique_names_gdf1, gdf2, similarity_threshold=0):
    """
    Update names in gdf2 based on similarity to names in gdf1.

    Parameters:
    - gdf1 (GeoDataFrame): First GeoDataFrame containing reference names.
    - gdf2 (GeoDataFrame): Second GeoDataFrame whose names need to be updated.
    - similarity_threshold (float): Threshold for similarity ratio. Default is 0.5.

    Returns:
    - None. Updates gdf2 in place.
    """
    # Iterate through rows of gdf2
    for index, row in gdf2.iterrows():
        # Get the value of the "NOM" column for the current row in gdf2
        name_gdf2 = row['NOM']
        highest_similarity_ratio = 0
        best_matching_name = None
        # Iterate through unique names in gdf1
        for name_gdf1 in unique_names_gdf1:
            # Calculate similarity ratio between names in gdf2 and gdf1
            similarity_ratio = SequenceMatcher(None, name_gdf1, name_gdf2).ratio()
            # Update best matching name if similarity ratio is higher
            if similarity_ratio > highest_similarity_ratio:
                highest_similarity_ratio = similarity_ratio
                best_matching_name = name_gdf1

        if highest_similarity_ratio >= similarity_threshold:
            # confirmation = input(f"Similarity found: '{name_gdf2}' -> '{name_gdf1}'Is this okay? (y/n): ").strip().lower()
            # if confirmation == "y":
            # print(f"{highest_similarity_ratio} for {name_gdf1} to {best_matching_name}")
            gdf2.at[index, 'NOM'] = best_matching_name

    return gdf2

def rasterize_geodataframe_by_column(gdf, column_name, resolution_x, resolution_y, output_path):
    # Get unique values from the specified column
    unique_values = sorted(gdf[column_name].unique())

    # Create a dictionary mapping unique values to unique identifiers
    value_to_index = {value: index + 1 for index, value in enumerate(unique_values)}

    # Add a new column to the GeoDataFrame containing the unique identifiers
    gdf['raster_value'] = gdf[column_name].map(value_to_index)

    # Get the bounds of the GeoDataFrame
    xmin, ymin, xmax, ymax = gdf.total_bounds

    # Calculate the number of pixels in x and y directions
    cols = int((xmax - xmin) / resolution_x)
    rows = int((ymax - ymin) / resolution_y)

    # Create a transform for the raster
    transform = from_origin(xmin, ymax, resolution_x, resolution_y)

    # Create an empty array to hold the rasterized values
    rasterized_array = np.zeros((rows, cols), dtype=np.uint8) # if bigger, change the dtype

    nodata_value = 255
    # Create a mask for nodata pixels

    # Rasterize each unique value separately
    for text_value, value in value_to_index.items():
        # print(f"Finished with {text_value}")
        mask = gdf['raster_value'] == value
        shapes = gdf.loc[mask, 'geometry']
        temp_raster = features.rasterize(
            shapes=shapes,
            out_shape=(rows, cols),
            # fill=nodata_value, # this covers areas that are not covered by geometries
            transform=transform,
            all_touched=True,
            default_value=value,
            dtype=np.uint8, # must be equal to the zeros
        )
        rasterized_array = np.maximum(rasterized_array, temp_raster)

    
    # Define the metadata for the raster
    profile = {
        'driver': 'GTiff',
        'height': rows,
        'width': cols,
        'count': 1,
        'dtype': rasterio.uint8,
        'crs': CRS.from_epsg(4326),
        'transform': transform,
        'nodata': nodata_value  # Set the nodata value in the profile metadata
    }

    # Write the raster array to a GeoTIFF file
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(rasterized_array, 1)

        # Set nodata values in the raster
        rasterized_array[rasterized_array == 0] = nodata_value
        dst.write(rasterized_array, 1)
        

In [8]:
gdf1 = gpd.read_file(vector_file_list[0])
gdf2 = gpd.read_file(vector_file_list[1])

# Create a mapping dictionary from gdf1
unique_names_gdf1 = gdf1['NOM'].unique().tolist()
unique_names_original_gdf2 = gdf2['NOM'].unique().tolist()

unique_names_gdf1 = gdf1['NOM'].unique().tolist()
print(unique_names_gdf1)

unique_names_gdf2 = gdf2['NOM'].unique().tolist()
print(unique_names_gdf2)

In [16]:
"""We are going to standarize the names of the column NOM since this will be the value which we will """
vector_file_list = get_vector_file_list(path)

names_list = ["Carrière Mine Infrastructure", "Cours d'eau", "Culture irriguée", "Culture maraîchère", "Culture pluviale", "Dune", "Forêt", "Forêt galerie", "Lac", "Localité", "Mangrove", "Mare", "Plaine inondable", "Plantation forestière", "Prairie aquatique", "Savane", "Savane arbustive", "Sol nu", "Steppe", "Tanne", "Vasière"]

output_path = r'Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat'

# Specify the column to rasterize by
column_name = 'NOM'

# Define the resolution of your raster
resolution_x = 50  # in meters
resolution_y = 50  # in meters

raster_path_list = []
for file in vector_file_list[1:]:
    output_path_file = os.path.join(output_path, os.path.basename(file).replace(".shp", "_1.tif"))
    raster_path_list.append(output_path_file)
    print(output_path_file)
    gdf = gpd.read_file(file)
    gdf = update_names_based_on_similarity(names_list, gdf, similarity_threshold=0.5)
    rasterize_geodataframe_by_column(gdf, column_name, resolution_x, resolution_y, output_path_file)

Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat\occsol_2015_anat_1.tif


Z:\z_resources\im-nca-senegal\v2_shp_occsol_anat\23-12-22\shp_occsol_anat\occsol_2020_anat_1.tif


["Cours d'eau", 'Carrière Mine Infrastructure', 'Lac', 'Mangrove', 'Culture irriguée', 'Plantation forestière', 'Prairie aquatique', 'Vasière', 'Forêt', 'Forêt galerie', 'Steppe', 'Plaine inondable', 'Mare', 'Dune', 'Culture maraîchère', 'Sol nu', 'Culture pluviale', 'Savane arbustive', 'Savane', 'Tanne', 'Localité']
